In [264]:
from relbench.datasets import get_dataset, get_dataset_names, register_dataset

In [ ]:
import os
import pandas as pd
import numpy as np
from relbench.base import Database, Dataset, Table

class TransactionalDataset(Dataset):
    # Set timestamps or other relevant information if needed
    val_timestamp = pd.Timestamp("2022-02-15")
    test_timestamp = pd.Timestamp("2022-02-22")

    def make_db(self) -> Database:
        # Path to your CSVs folder
        path = os.path.join("C:/Users/KN2C/Desktop/Dani/relbench/relbench/", "hyper_data")
        customers = os.path.join(path, "Customers.csv")
        articles = os.path.join(path, "Articles.csv")
        branches = os.path.join(path, "Branches.csv")
        transactions = os.path.join(path, "Transactions.csv")

        # Ensure that CSV files exist in the specified path
        if not os.path.exists(customers):
            raise RuntimeError(f"Dataset not found at '{path}'. Please make sure the CSV files are in the correct folder.")

        # Read the CSV data into pandas DataFrames
        customers_df = pd.read_csv(customers)
        articles_df = pd.read_csv(articles)
        branches_df = pd.read_csv(branches)
        transactions_df = pd.read_csv(transactions)

        ################################################################################
        # Check for and handle duplicate primary keys in articles, customers, and branches tables
        ################################################################################

        # Handle duplicates in the articles table
        if articles_df.duplicated(subset=['articles_id']).any():
            print("Duplicates found in the 'articles_id' column. Removing duplicates...")
            articles_df = articles_df.drop_duplicates(subset=['articles_id'], keep='first')

        # Handle duplicates in the customers table
        if customers_df.duplicated(subset=['customers_id']).any():
            print("Duplicates found in the 'customers_id' column. Removing duplicates...")
            customers_df = customers_df.drop_duplicates(subset=['customers_id'], keep='first')

        # Handle duplicates in the branches table
        if branches_df.duplicated(subset=['BranchCode']).any():
            print("Duplicates found in the 'BranchCode' column. Removing duplicates...")
            branches_df = branches_df.drop_duplicates(subset=['BranchCode'], keep='first')

        ################################################################################
        # Clean and process the data (drop unnecessary columns, handle missing data)
        ################################################################################
        # Drop unnecessary columns
        transactions_df.drop(columns=["Return Amount"], inplace=True)
        articles_df.drop(columns=["Item Barcode", "External Item Number"], inplace=True)

        # Replace any missing or invalid values
        transactions_df["salesTime"] = transactions_df["salesTime"].replace(r"^\\N$", "00:00:00", regex=True)
        transactions_df = transactions_df.replace(r"^\\N$", np.nan, regex=True)

        # Combine date and time into a single 'datetime' column
        # transactions_df['datetime'] = pd.to_datetime(transactions_df['d_dat'] + ' ' + transactions_df['salesTime'])
        # transactions_df.drop(columns=["d_dat"], inplace=True)        
        # Convert date column to pd.Timestamp
        # transactions_df["datetime"] = pd.to_datetime(transactions_df["datetime"])

        transactions_df["datetime"] = pd.to_datetime(
        transactions_df["d_dat"], format="%Y-%m-%d"
        )
        transactions_df.drop(columns=["d_dat"], inplace=True)          
        # Convert other fields if necessary
        transactions_df['price_purchase'] = pd.to_numeric(transactions_df['price_purchase'], errors='coerce')
        transactions_df['Discount_ratio'] = pd.to_numeric(transactions_df['Discount_ratio'], errors='coerce')
        transactions_df['Quantity'] = pd.to_numeric(transactions_df['Quantity'], errors='coerce')

        ################################################################################
        # Now we define the table structure and relationships.
        ################################################################################

        tables = {}

        # Articles table
        tables["article"] = Table(
            df=pd.DataFrame(articles_df),
            fkey_col_to_pkey_table={},
            pkey_col="articles_id",
            time_col=None,
        )

        # Customers table
        tables["customer"] = Table(
            df=pd.DataFrame(customers_df),
            fkey_col_to_pkey_table={},
            pkey_col="customers_id",
            time_col=None,
        )

        # Branches table (renamed from "branche" to "branches")
        tables["branches"] = Table(
            df=pd.DataFrame(branches_df),
            fkey_col_to_pkey_table={},
            pkey_col="BranchCode",
            time_col=None,
        )

        # Transactions table
        tables["transactions"] = Table(
            df=pd.DataFrame(transactions_df),
            fkey_col_to_pkey_table={
                "articles_id": "article",    # Foreign key to articles
                "customers_id": "customer",  # Foreign key to customers
                "BranchCode": "branches",    # Foreign key to branches
            },
            pkey_col=None,
            time_col="datetime",  # Use the combined datetime column for time-based operations
        )

        return Database(tables)


In [265]:
import os
import pandas as pd
import numpy as np
from relbench.base import Database, Dataset, Table

class TransactionalDataset(Dataset):
    # Set timestamps or other relevant information if needed
    val_timestamp = pd.Timestamp("2024-07-18")
    test_timestamp = pd.Timestamp("2024-07-25")

    def make_db(self) -> Database:
        # Path to your CSVs folder
        path = os.path.join("C:/Users/KN2C/Desktop/Dani/relbench/relbench/", "burger_data")
        customers = os.path.join(path, "customer_data.csv")
        articles = os.path.join(path, "article_data.csv")
        branches = os.path.join(path, "branch_data.csv")
        transactions = os.path.join(path, "transaction_data.csv")

        # Ensure that CSV files exist in the specified path
        if not os.path.exists(customers):
            raise RuntimeError(f"Dataset not found at '{path}'. Please make sure the CSV files are in the correct folder.")

        # Read the CSV data into pandas DataFrames
        customers_df = pd.read_csv(customers)
        articles_df = pd.read_csv(articles)
        branches_df = pd.read_csv(branches)
        transactions_df = pd.read_csv(transactions)
        transactions_df['d_dat'] = pd.to_datetime(transactions_df['d_dat'])
        split_date = pd.to_datetime('2022-01-01')
        transactions_df = transactions_df[transactions_df['d_dat'] >= split_date]
        transactions_df = transactions_df.reset_index(drop=True)
        ################################################################################
        # Check for and handle duplicate primary keys in articles, customers, and branches tables
        ################################################################################

        # Handle duplicates in the articles table
        if articles_df.duplicated(subset=['articles_id']).any():
            print("Duplicates found in the 'articles_id' column. Removing duplicates...")
            articles_df = articles_df.drop_duplicates(subset=['articles_id'], keep='first')

        # Handle duplicates in the customers table
        if customers_df.duplicated(subset=['customers_id']).any():
            print("Duplicates found in the 'customers_id' column. Removing duplicates...")
            customers_df = customers_df.drop_duplicates(subset=['customers_id'], keep='first')

        # Handle duplicates in the branches table
        if branches_df.duplicated(subset=['BranchCode']).any():
            print("Duplicates found in the 'BranchCode' column. Removing duplicates...")
            branches_df = branches_df.drop_duplicates(subset=['BranchCode'], keep='first')

        ################################################################################
        # Clean and process the data (drop unnecessary columns, handle missing data)
        ################################################################################
        # Drop unnecessary columns
        transactions_df.drop(columns=["factor_id"], inplace=True)
        transactions_df.drop(columns=["date"], inplace=True)
        transactions_df.drop(columns=["art_name"], inplace=True)
        transactions_df.drop(columns=["price_item"], inplace=True)
        transactions_df.drop(columns=["BranchName"], inplace=True)
        transactions_df.drop(columns=["gerd"], inplace=True)
        transactions_df.drop(columns=["Maliat"], inplace=True)        
        transactions_df.drop(columns=["group_id"], inplace=True)            
        # articles_df.drop(columns=["Item Barcode", "External Item Number"], inplace=True)

        transactions_df['Payment channel'] = pd.factorize(transactions_df['Payment channel'])[0]

        # Encoding Payment channel column
        transactions_df['week'] = pd.factorize(transactions_df['week'])[0]


        # Replace any missing or invalid values
        # transactions_df["salesTime"] = transactions_df["salesTime"].replace(r"^\\N$", "00:00:00", regex=True)
        transactions_df = transactions_df.replace(r"^\\N$", np.nan, regex=True)

        # Combine date and time into a single 'datetime' column
        # transactions_df['datetime'] = pd.to_datetime(transactions_df['d_dat'] + ' ' + transactions_df['salesTime'])
        # transactions_df.drop(columns=["d_dat"], inplace=True)        
        # Convert date column to pd.Timestamp
        # transactions_df["datetime"] = pd.to_datetime(transactions_df["datetime"])

        transactions_df["datetime"] = pd.to_datetime(
        transactions_df["d_dat"], format="%Y-%m-%d"
        )
        transactions_df.drop(columns=["d_dat"], inplace=True)          
        # Convert other fields if necessary
        transactions_df['price_purchase'] = pd.to_numeric(transactions_df['price_purchase'], errors='coerce')
        transactions_df['takhfif'] = pd.to_numeric(transactions_df['takhfif'], errors='coerce')
        transactions_df['Quantity'] = pd.to_numeric(transactions_df['Quantity'], errors='coerce')

        ################################################################################
        # Now we define the table structure and relationships.
        ################################################################################

        tables = {}

        # Articles table
        tables["article"] = Table(
            df=pd.DataFrame(articles_df),
            fkey_col_to_pkey_table={},
            pkey_col="articles_id",
            time_col=None,
        )

        # Customers table
        tables["customer"] = Table(
            df=pd.DataFrame(customers_df),
            fkey_col_to_pkey_table={},
            pkey_col="customers_id",
            time_col=None,
        )

        # Branches table (renamed from "branche" to "branches")
        tables["branches"] = Table(
            df=pd.DataFrame(branches_df),
            fkey_col_to_pkey_table={},
            pkey_col="BranchCode",
            time_col=None,
        )

        # Transactions table
        tables["transactions"] = Table(
            df=pd.DataFrame(transactions_df),
            fkey_col_to_pkey_table={
                "articles_id": "article",    # Foreign key to articles
                "customers_id": "customer",  # Foreign key to customers
                "BranchCode": "branches",    # Foreign key to branches
            },
            pkey_col=None,
            time_col="datetime",  # Use the combined datetime column for time-based operations
        )

        return Database(tables)


In [266]:
transactional_dataset = TransactionalDataset()
db = transactional_dataset.make_db()

Duplicates found in the 'articles_id' column. Removing duplicates...


In [247]:
table = db.table_dict["transactions"]

In [199]:
original = table.df

In [201]:
original.to_csv('C:\\Users\\KN2C\\Desktop\\Dani\\relbench\\original.csv',index=False)

In [258]:
db.table_dict["branches"]

Table(df=
    BranchCode          BranchName
0        13803            پاسداران
1        13804             اندرزگو
2        13802           تهرانپارس
3        13808           باغ فردوس
4        13805         سعادت آباد 
5        13809                کورش
6        13807         هايپر استار
7        13806               توچال
8        13800            کال سنتر
9        13811               بيهقي
10       13810             جم سنتر
11       13812            هديش مال
12       13813             ستارخان
13       13814  شعبه بازار (پرنسا),
  fkey_col_to_pkey_table={},
  pkey_col=BranchCode,
  time_col=None)

In [249]:
db.table_dict["article"]

Table(df=
      articles_id                art_name  price_item  group_id
0             611            خانواده پپسي        2500       600
1             501         سيب زميني مخصوص        7500       500
2             101                لند برگر       13500       100
3             102             اسموكي برگر       14500       100
4             601               پپسي قوطي        1500       600
...           ...                     ...         ...       ...
5546          324      سوپر پيتزا چهارفصل      189500       320
5547          322  سوپر پيتزا چيکن آلفردو      179500       320
5548          323      سوپر پيتزا رست بيف      185500       320
6044         1138        سوپر برگر ماشروم      212500       201
6045         1137               سوپر برگر      209500       201

[209 rows x 4 columns],
  fkey_col_to_pkey_table={},
  pkey_col=articles_id,
  time_col=None)

In [250]:
db.table_dict["customer"]

Table(df=
        customers_id  total_spend
0         2122248666       317000
1         2122378721       650000
2         2122480299       377000
3         2122601851      3363875
4         2122682724       950500
...              ...          ...
864876    9999995223      3560000
864877    9999996611      1336400
864878    9999999909       144500
864879    9999999994       999000
864880    9999999999     18477800

[864881 rows x 2 columns],
  fkey_col_to_pkey_table={},
  pkey_col=customers_id,
  time_col=None)

In [251]:
table.df.iloc[table.df["datetime"].idxmax()]


SaleTime                      16:40:32
articles_id                        602
price_purchase                 24000.0
Quantity                           1.0
BranchCode                       13814
customers_id                9182309725
takhfif                            0.0
Payment channel                      2
Sales channel                     اسنپ
week                               135
price_basket                  309500.0
datetime           2024-08-02 00:00:00
Name: 4638308, dtype: object

In [209]:
table.df.iloc[table.df["datetime"].idxmin()]

SaleTime                      23:29:45
articles_id                        303
price_purchase                 99500.0
Quantity                           1.0
BranchCode                       13808
customers_id                9120800148
takhfif                            0.0
Payment channel                      0
Sales channel                    حضوری
week                                 0
price_basket                  155200.0
datetime           2022-01-01 00:00:00
Name: 0, dtype: object

In [267]:
register_dataset("burger-aras7", TransactionalDataset)
get_dataset_names()

['rel-amazon',
 'rel-avito',
 'rel-event',
 'rel-f1',
 'rel-hm',
 'rel-stack',
 'rel-trial',
 'burger-aras',
 'burger-aras1',
 'burger-aras2',
 'burger-aras3',
 'burger-aras4',
 'burger-aras5',
 'burger-aras6',
 'burger-aras7']

In [268]:
hyper_dataset = get_dataset("burger-aras7")
hyper_dataset

TransactionalDataset()

In [269]:
hyper_dataset.val_timestamp, hyper_dataset.test_timestamp

(Timestamp('2024-07-18 00:00:00'), Timestamp('2024-07-25 00:00:00'))

In [270]:
import relbench

relbench.__version__

'1.1.0'

In [271]:
import duckdb
import pandas as pd
from relbench.tasks import get_task, get_task_names, register_task
from relbench.base import Database, EntityTask, RecommendationTask, Table, TaskType
from relbench.metrics import (
    accuracy,
    average_precision,
    f1,
    link_prediction_map,
    link_prediction_precision,
    link_prediction_recall,
    mae,
    r2,
    rmse,
    roc_auc,
)
from metrics import link_prediction_top
class UserItemPurchaseTask(RecommendationTask):
    r"""Predict the list of articles each customer will purchase in the next seven
    days."""

    task_type = TaskType.LINK_PREDICTION
    src_entity_col = "customer_id"
    src_entity_table = "customer"
    dst_entity_col = "article_id"
    dst_entity_table = "article"
    time_col = "timestamp"
    timedelta = pd.Timedelta(days=7)
    metrics = [link_prediction_precision, link_prediction_recall, link_prediction_map, link_prediction_top]
    eval_k = 12

    def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
        customer = db.table_dict["customer"].df
        transactions = db.table_dict["transactions"].df
        timestamp_df = pd.DataFrame({"timestamp": timestamps})

        df = duckdb.sql(
            f"""
            SELECT
                t.timestamp,
                transactions.customer_id,
                LIST(DISTINCT transactions.article_id) AS article_id
            FROM
                timestamp_df t
            LEFT JOIN
                transactions
            ON
                transactions.t_dat > t.timestamp AND
                transactions.t_dat <= t.timestamp + INTERVAL '{self.timedelta} days'
            GROUP BY
                t.timestamp,
                transactions.customer_id
            """
        ).df()

        return Table(
            df=df,
            fkey_col_to_pkey_table={
                self.src_entity_col: self.src_entity_table,
                self.dst_entity_col: self.dst_entity_table,
            },
            pkey_col=None,
            time_col=self.time_col,
        )

# Task 1: Predict articles each customer will purchase in the next 7 days
# class CustomerArticlePurchaseTask(RecommendationTask):
#     r"""Predict the list of articles each customer will purchase in the next seven days."""
    
#     task_type = TaskType.LINK_PREDICTION
#     src_entity_col = "customers_id"
#     src_entity_table = "customer"
#     dst_entity_col = "articles_id"
#     dst_entity_table = "article"
#     time_col = "timestamp"
#     timedelta = pd.Timedelta(days=7)
#     metrics = [link_prediction_precision, link_prediction_recall, link_prediction_map, link_prediction_top]
#     eval_k = 4

#     def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
#         transactions = db.table_dict["transactions"].df
#         timestamp_df = pd.DataFrame({"timestamp": timestamps})

#         df = duckdb.sql(
#             f"""
#             SELECT
#                 t.timestamp,
#                 transactions.customers_id,
#                 LIST(DISTINCT transactions.articles_id) AS articles_id
#             FROM
#                 timestamp_df t
#             LEFT JOIN
#                 transactions
#             ON
#                 transactions.datetime > t.timestamp AND
#                 transactions.datetime <= t.timestamp + INTERVAL '{self.timedelta.days} days'
#             GROUP BY
#                 t.timestamp,
#                 transactions.customers_id
#             """
#         ).df()

#         return Table(
#             df=df,
#             fkey_col_to_pkey_table={
#                 self.src_entity_col: self.src_entity_table,
#                 self.dst_entity_col: self.dst_entity_table,
#             },
#             pkey_col=None,
#             time_col=self.time_col,
#         )


# Task 2: Predict customer churn (no purchases in the next week)
class CustomerChurnTask(EntityTask):
    r"""Predict the churn for a customer (no transactions) in the next 6 days."""

    task_type = TaskType.BINARY_CLASSIFICATION
    entity_col = "customers_id"
    entity_table = "customer"
    time_col = "timestamp"
    target_col = "churn"
    timedelta = pd.Timedelta(days=7)
    metrics = [average_precision, accuracy, f1, roc_auc]

    def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
        customer = db.table_dict["customer"].df
        transactions = db.table_dict["transactions"].df
        timestamp_df = pd.DataFrame({"timestamp": timestamps})

        df = duckdb.sql(
            f"""
            SELECT
                timestamp,
                customers_id,
                CAST(
                    NOT EXISTS (
                        SELECT 1
                        FROM transactions
                        WHERE
                            transactions.customers_id = customer.customers_id AND
                            transactions.datetime > timestamp AND
                            transactions.datetime <= timestamp + INTERVAL '{self.timedelta}'
                    ) AS INTEGER
                ) AS churn
            FROM
                timestamp_df,
                customer
            WHERE
                EXISTS (
                    SELECT 1
                    FROM transactions
                    WHERE
                        transactions.customers_id = customer.customers_id AND
                        transactions.datetime > timestamp - INTERVAL '{self.timedelta}' AND
                        transactions.datetime <= timestamp
                )
            """
        ).df()

        return Table(
            df=df,
            fkey_col_to_pkey_table={self.entity_col: self.entity_table},
            pkey_col=None,
            time_col=self.time_col,
        ) 
    # def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
    #     transactions = db.table_dict["transactions"].df
    #     customer = db.table_dict["customer"].df
    #     timestamp_df = pd.DataFrame({"timestamp": timestamps})

    #     df = duckdb.sql(
    #         f"""
    #         SELECT
    #             t.timestamp,
    #             c.customers_id,
    #             CAST(
    #                 NOT EXISTS (
    #                     SELECT 1
    #                     FROM transactions
    #                     WHERE
    #                         transactions.customers_id = c.customers_id AND
    #                         transactions.datetime > t.timestamp AND
    #                         transactions.datetime <= t.timestamp + INTERVAL '{self.timedelta.days} days'
    #                 ) AS INTEGER
    #             ) AS churn
    #         FROM
    #             timestamp_df t,
    #             customer c
    #         WHERE
    #             EXISTS (
    #                 SELECT 1
    #                 FROM transactions
    #                 WHERE
    #                     transactions.customers_id = c.customers_id AND
    #                     transactions.datetime > t.timestamp - INTERVAL '{self.timedelta.days} days' AND
    #                     transactions.datetime <= t.timestamp
    #             )
    #         """
    #     ).df()

    #     return Table(
    #         df=df,
    #         fkey_col_to_pkey_table={self.entity_col: self.entity_table},
    #         pkey_col=None,
    #         time_col=self.time_col,
    #     )


# Task 3: Predict article sales in the next 7 days
class ArticleSalesTask(EntityTask):
    r"""Predict the total sales for an article (sum of `price_purchase`) in the next 7 days."""
    
    task_type = TaskType.REGRESSION
    entity_col = "articles_id"
    entity_table = "article"
    time_col = "datetime"
    target_col = "sales"
    timedelta = pd.Timedelta(days=7)
    metrics = [r2, mae, rmse]

    def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
        transactions = db.table_dict["transactions"].df
        articles = db.table_dict["article"].df
        timestamp_df = pd.DataFrame({"timestamp": timestamps})

        df = duckdb.sql(
            f"""
            SELECT
                t.timestamp,
                a.articles_id,
                COALESCE(SUM(transactions.price_purchase), 0) AS sales
            FROM
                timestamp_df t,
                article a
            LEFT JOIN
                transactions
            ON
                transactions.articles_id = a.articles_id AND
                transactions.datetime > t.timestamp AND
                transactions.datetime <= t.timestamp + INTERVAL '{self.timedelta.days} days'
            GROUP BY
                t.timestamp,
                a.articles_id
            """
        ).df()

        return Table(
            df=df,
            fkey_col_to_pkey_table={self.entity_col: self.entity_table},
            pkey_col=None,
            time_col=self.time_col,
        )



In [272]:
customer = db.table_dict["customer"].df

In [273]:
print(customer)

        customers_id  total_spend
0         2122248666       317000
1         2122378721       650000
2         2122480299       377000
3         2122601851      3363875
4         2122682724       950500
...              ...          ...
864876    9999995223      3560000
864877    9999996611      1336400
864878    9999999909       144500
864879    9999999994       999000
864880    9999999999     18477800

[864881 rows x 2 columns]


In [274]:
aras_churn_task = CustomerChurnTask(hyper_dataset, cache_dir="./cache/burger_aras35655")
aras_churn_task

CustomerChurnTask(dataset=TransactionalDataset())

In [276]:
register_task("burger-aras7", "aras_churn_task1", CustomerChurnTask)
get_task_names("burger-aras7")

['aras_churn_task1']

In [277]:
get_task_names("burger-aras7")

['aras_churn_task1']

In [278]:
import numpy as np

from torch.nn import BCEWithLogitsLoss, L1Loss
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("burger-aras7")
task = get_task("burger-aras7", "aras_churn_task1")


train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test")

out_channels = 1
loss_fn = BCEWithLogitsLoss()
tune_metric = "roc_auc"
higher_is_better = True

Making task table for train split from scratch...
(You can also use `get_task(..., download=True)` for tasks prepared by the RelBench team.)
Making Database object from scratch...
(You can also use `get_dataset(..., download=True)` for datasets prepared by the RelBench team.)
Duplicates found in the 'articles_id' column. Removing duplicates...
Done in 46.72 seconds.
Caching Database object to C:\Users\KN2C\AppData\Local\relbench\relbench\Cache/burger-aras7/db...
Done in 1.83 seconds.
Loading Database object from C:\Users\KN2C\AppData\Local\relbench\relbench\Cache/burger-aras7/db...
Done in 0.80 seconds.
Done in 92.71 seconds.
Making task table for val split from scratch...
(You can also use `get_task(..., download=True)` for tasks prepared by the RelBench team.)
Done in 0.27 seconds.
Making task table for test split from scratch...
(You can also use `get_task(..., download=True)` for tasks prepared by the RelBench team.)
Loading Database object from C:\Users\KN2C\AppData\Local\relbench

In [280]:
original_to_new_map = db.reindex_pkeys_and_fkeys()

In [285]:
print(original_to_new_map)

None


In [283]:
reverse_map = {v: k for k, v in original_to_new_map['df'].items()}
train_table['original_customers_id'] = train_table['customers_id'].map(reverse_map)

TypeError: 'NoneType' object is not subscriptable

In [279]:
train_table

Table(df=
         timestamp  customers_id  churn
0       2022-09-01        820788      1
1       2022-09-01        820860      1
2       2022-08-25        821042      1
3       2022-08-18        820248      1
4       2022-08-11        819444      1
...            ...           ...    ...
1504457 2022-01-20        306144      1
1504458 2022-01-20        306755      1
1504459 2022-01-13        305514      1
1504460 2022-01-13        306885      1
1504461 2022-01-13        307196      1

[1504462 rows x 3 columns],
  fkey_col_to_pkey_table={'customers_id': 'customer'},
  pkey_col=None,
  time_col=timestamp)

In [157]:
val_table

Table(df=
      timestamp  customers_id  churn
0    2024-07-18          2375      1
1    2024-07-18          3591      1
2    2024-07-18          5906      1
3    2024-07-18          6873      1
4    2024-07-18          7290      1
...         ...           ...    ...
9720 2024-07-18        199165      1
9721 2024-07-18        199695      1
9722 2024-07-18        202659      1
9723 2024-07-18        202791      1
9724 2024-07-18        203882      1

[9725 rows x 3 columns],
  fkey_col_to_pkey_table={'customers_id': 'customer'},
  pkey_col=None,
  time_col=timestamp)

In [158]:
test_table

Table(df=
       timestamp  customers_id
0     2024-07-25        820105
1     2024-07-25        820432
2     2024-07-25        820663
3     2024-07-25        822402
4     2024-07-25        824978
...          ...           ...
10989 2024-07-25        600080
10990 2024-07-25        603426
10991 2024-07-25        607081
10992 2024-07-25        608889
10993 2024-07-25        609116

[10994 rows x 2 columns],
  fkey_col_to_pkey_table={'customers_id': 'customer'},
  pkey_col=None,
  time_col=timestamp)

In [159]:
import os
import math
import numpy as np
from tqdm import tqdm

import torch
import torch_geometric
import torch_frame

# Some book keeping
from torch_geometric.seed import seed_everything

seed_everything(42)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # check that it's cuda if you want it to run in reasonable time!
root_dir = "./data_ARAS"

cuda


In [160]:
from relbench.modeling.utils import get_stype_proposal

db = dataset.get_db()
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict

{'article': {'articles_id': <stype.numerical: 'numerical'>,
  'art_name': <stype.text_embedded: 'text_embedded'>,
  'price_item': <stype.numerical: 'numerical'>,
  'group_id': <stype.numerical: 'numerical'>},
 'branches': {'BranchCode': <stype.numerical: 'numerical'>,
  'BranchName': <stype.text_embedded: 'text_embedded'>},
 'customer': {'customers_id': <stype.numerical: 'numerical'>,
  'total_spend': <stype.numerical: 'numerical'>},
 'transactions': {'SaleTime': <stype.timestamp: 'timestamp'>,
  'articles_id': <stype.numerical: 'numerical'>,
  'price_purchase': <stype.numerical: 'numerical'>,
  'Quantity': <stype.numerical: 'numerical'>,
  'BranchCode': <stype.numerical: 'numerical'>,
  'customers_id': <stype.numerical: 'numerical'>,
  'takhfif': <stype.numerical: 'numerical'>,
  'Payment channel': <stype.categorical: 'categorical'>,
  'Sales channel': <stype.categorical: 'categorical'>,
  'week': <stype.numerical: 'numerical'>,
  'price_basket': <stype.numerical: 'numerical'>,
  'dat

In [161]:
from typing import List, Optional
from sentence_transformers import SentenceTransformer
from torch import Tensor
import torch

class BertPersianTextEmbedding:
    def __init__(self, device: Optional[torch.device] = None):
        # Replace the model with a Persian BERT model
        self.model = SentenceTransformer("HooshvareLab/bert-fa-zwnj-base",  # Example Persian BERT model
            device=device,
        )

    def __call__(self, sentences: List[str]) -> Tensor:
        # Encode the sentences using the Persian BERT model and return as a tensor
        return torch.from_numpy(self.model.encode(sentences))


In [14]:
from typing import List, Optional
from sentence_transformers import SentenceTransformer
from torch import Tensor


class GloveTextEmbedding:
    def __init__(self, device: Optional[torch.device
                                       ] = None):
        self.model = SentenceTransformer(
            "sentence-transformers/average_word_embeddings_glove.6B.300d",
            device=device,
        )

    def __call__(self, sentences: List[str]) -> Tensor:
        return torch.from_numpy(self.model.encode(sentences))


In [24]:
from torch_frame.config.text_embedder import TextEmbedderConfig
from relbench.modeling.graph import make_pkey_fkey_graph

text_embedder_cfg = TextEmbedderConfig(
    text_embedder=GloveTextEmbedding(device=device), batch_size=256
)

data, col_stats_dict = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=col_to_stype_dict,  # speficied column types
    text_embedder_cfg=text_embedder_cfg,  # our chosen text encoder
    cache_dir=os.path.join(
        root_dir, f"rel-hm_materialized_cache1"
    ),  # store materialized graph for convenience
)

Embedding raw data in mini-batch: 100%|██████████| 5360/5360 [00:34<00:00, 153.40it/s]


RuntimeError: [enforce fail at inline_container.cc:595] . unexpected pos 38417024 vs 38416960

In [98]:
torch.cuda.empty_cache()

In [162]:
from torch_frame.config.text_embedder import TextEmbedderConfig
from relbench.modeling.graph import make_pkey_fkey_graph

text_embedder_cfg = TextEmbedderConfig(
    text_embedder=BertPersianTextEmbedding(device=device), batch_size=64
)

data, col_stats_dict = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=col_to_stype_dict,  # speficied column types
    text_embedder_cfg=text_embedder_cfg,  # our chosen text encoder
    cache_dir=os.path.join(
        root_dir, f"rel-aras_churn_materialized_cache3"
    ),  # store materialized graph for convenience
)

Some weights of BertModel were not initialized from the model checkpoint at HooshvareLab/bert-fa-zwnj-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [163]:
data

HeteroData(
  article={ tf=TensorFrame([209, 3]) },
  branches={ tf=TensorFrame([14, 1]) },
  customer={ tf=TensorFrame([864881, 1]) },
  transactions={
    tf=TensorFrame([4603168, 9]),
    time=[4603168],
  },
  (transactions, f2p_articles_id, article)={ edge_index=[2, 4603168] },
  (article, rev_f2p_articles_id, transactions)={ edge_index=[2, 4603168] },
  (transactions, f2p_customers_id, customer)={ edge_index=[2, 4603168] },
  (customer, rev_f2p_customers_id, transactions)={ edge_index=[2, 4603168] },
  (transactions, f2p_BranchCode, branches)={ edge_index=[2, 4603168] },
  (branches, rev_f2p_BranchCode, transactions)={ edge_index=[2, 4603168] }
)

In [164]:
data["customer"].tf

TensorFrame(
  num_cols=1,
  num_rows=864881,
  numerical (1): ['total_spend'],
  has_target=False,
  device='cpu',
)

In [165]:
data["article"].tf

TensorFrame(
  num_cols=3,
  num_rows=209,
  numerical (2): ['group_id', 'price_item'],
  embedding (1): ['art_name'],
  has_target=False,
  device='cpu',
)

In [166]:
data["transactions"].tf

TensorFrame(
  num_cols=9,
  num_rows=4603168,
  timestamp (2): ['SaleTime', 'datetime'],
  numerical (5): ['Quantity', 'price_basket', 'price_purchase', 'takhfif', 'week'],
  categorical (2): ['Payment channel', 'Sales channel'],
  has_target=False,
  device='cpu',
)

In [167]:
list(data["customer"].keys())

['tf']

In [168]:
data["customer"].tf[10]

TensorFrame(
  num_cols=1,
  num_rows=1,
  numerical (1): ['total_spend'],
  has_target=False,
  device='cpu',
)

In [169]:
data[("transactions", "f2p_customers_id", "customer")]

{'edge_index': tensor([[      0,       1,       2,  ..., 4603165, 4603166, 4603167],
        [ 115012,  115012,  115012,  ...,  131742,  131742,  423398]])}

In [22]:
from typing import Any, Dict, List

import torch
from torch import Tensor
from torch.nn import Embedding, ModuleDict
from torch_frame.data.stats import StatType
from torch_geometric.data import HeteroData
from torch_geometric.nn import MLP
from torch_geometric.typing import NodeType

from relbench.modeling.nn import HeteroEncoder, HeteroGraphSAGE, HeteroTemporalEncoder


class Model(torch.nn.Module):

    def __init__(
        self,
        data: HeteroData,
        col_stats_dict: Dict[str, Dict[str, Dict[StatType, Any]]],
        num_layers: int,
        channels: int,
        out_channels: int,
        aggr: str,
        norm: str,
        # List of node types to add shallow embeddings to input
        shallow_list: List[NodeType] = [],
        # ID awareness
        id_awareness: bool = False,
    ):
        super().__init__()

        self.encoder = HeteroEncoder(
            channels=channels,
            node_to_col_names_dict={
                node_type: data[node_type].tf.col_names_dict
                for node_type in data.node_types
            },
            node_to_col_stats=col_stats_dict,
        )
        self.temporal_encoder = HeteroTemporalEncoder(
            node_types=[
                node_type for node_type in data.node_types if "time" in data[node_type]
            ],
            channels=channels,
        )
        self.gnn = HeteroGraphSAGE(
            node_types=data.node_types,
            edge_types=data.edge_types,
            channels=channels,
            aggr=aggr,
            num_layers=num_layers,
        )
        self.head = MLP(
            channels,
            out_channels=out_channels,
            norm=norm,
            num_layers=1,
        )
        self.embedding_dict = ModuleDict(
            {
                node: Embedding(data.num_nodes_dict[node], channels)
                for node in shallow_list
            }
        )

        self.id_awareness_emb = None
        if id_awareness:
            self.id_awareness_emb = torch.nn.Embedding(1, channels)
        self.reset_parameters()

    def reset_parameters(self):
        self.encoder.reset_parameters()
        self.temporal_encoder.reset_parameters()
        self.gnn.reset_parameters()
        self.head.reset_parameters()
        for embedding in self.embedding_dict.values():
            torch.nn.init.normal_(embedding.weight, std=0.1)
        if self.id_awareness_emb is not None:
            self.id_awareness_emb.reset_parameters()

    def forward(
        self,
        batch: HeteroData,
        entity_table: NodeType,
    ) -> Tensor:
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
            batch.num_sampled_nodes_dict,
            batch.num_sampled_edges_dict,
        )

        return self.head(x_dict[entity_table][: seed_time.size(0)])

    def forward_dst_readout(
        self,
        batch: HeteroData,
        entity_table: NodeType,
        dst_table: NodeType,
    ) -> Tensor:
        if self.id_awareness_emb is None:
            raise RuntimeError(
                "id_awareness must be set True to use forward_dst_readout"
            )
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)
        # Add ID-awareness to the root node
        x_dict[entity_table][: seed_time.size(0)] += self.id_awareness_emb.weight

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
        )

        return self.head(x_dict[dst_table])

In [23]:
import argparse
import copy
import json
import os
import warnings
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn.functional as F
# from model import Model
# from text_embedder import GloveTextEmbedding
from torch import Tensor
from torch_frame import stype
from torch_frame.config.text_embedder import TextEmbedderConfig
from torch_geometric.loader import NeighborLoader
from torch_geometric.seed import seed_everything
from tqdm import tqdm

from relbench.base import Dataset, RecommendationTask, TaskType
from relbench.datasets import get_dataset
from relbench.modeling.graph import get_link_train_table_input, make_pkey_fkey_graph
from relbench.modeling.loader import SparseTensor
from relbench.modeling.utils import get_stype_proposal
from relbench.tasks import get_task

In [24]:
# Initialize the loader dictionary
loader_dict: Dict[str, NeighborLoader] = {}
dst_nodes_dict: Dict[str, Tuple[NodeType, Tensor]] = {}

# Loop over the train, val, and test splits
for split, table in [
    ("train", train_table),
    ("val", val_table),
    ("test", test_table),
]:
    # Get link train table input for link prediction task
    table_input = get_link_train_table_input(
        table=table,
        task=task,
    )
    
    # Save destination nodes for later use
    dst_nodes_dict[split] = table_input.dst_nodes

    # Create NeighborLoader for link prediction
    loader_dict[split] = NeighborLoader(
        data,
        num_neighbors=[128 for _ in range(2)],  # Sample subgraphs of depth 2, 128 neighbors per node
        time_attr="time",  # Use time attribute if available
        input_nodes=table_input.src_nodes,  # Source nodes for link prediction
        input_time=table_input.src_time,  # Use src_time if time data is available
        subgraph_type="bidirectional",
        batch_size=512,
        temporal_strategy="last",  # Uniform sampling strategy for time
        shuffle=split == "train",  # Shuffle only during training
        num_workers=0,
        persistent_workers=False,
    )


c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\relbench\modeling\graph.py:217: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  dst_node_indices = sparse_coo.to_sparse_csr()


In [25]:
# Initialize the model for link prediction task
model = Model(
    data=data,  # Heterogeneous data object
    col_stats_dict=col_stats_dict,  # Column statistics dictionary
    num_layers=2,  # Adjust this to match your desired architecture (depth of GNN)
    channels=128,  # Number of hidden channels in GNN layers
    out_channels=1,  # Output size (for link prediction, usually a scalar per edge)
    aggr="sum",  # Aggregation method (can be "sum", "mean", etc.)
    norm="layer_norm",  # Normalization method
    id_awareness=True,  # Whether the model is aware of node IDs
).to(device)  # Move model to the appropriate device (e.g., GPU)

# Set up the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Use the desired learning rate

# Handling sparse destination nodes for training
# dst_nodes_dict stores the destination nodes for the "train" split (in sparse format)
train_sparse_tensor = SparseTensor(dst_nodes_dict["train"][1], device=device)


In [26]:
def train() -> float:
    model.train()  # Set model to training mode

    loss_accum = count_accum = 0
    steps = 0
    total_steps = min(len(loader_dict["train"]), 2000)  # Change the max_steps_per_epoch to 2000 or your preferred value

    for batch in tqdm(loader_dict["train"], total=total_steps):
        batch = batch.to(device)  # Move batch data to device (GPU or CPU)

        # Forward pass through the model for link prediction (source and destination tables)
        out = model.forward_dst_readout(
            batch, task.src_entity_table, task.dst_entity_table
        ).flatten()  # Flatten the output

        batch_size = batch[task.src_entity_table].batch_size  # Get batch size for the source entity table

        # Get ground-truth labels
        input_id = batch[task.src_entity_table].input_id  # Input IDs for the batch
        src_batch, dst_index = train_sparse_tensor[input_id]  # Get the source and destination indices

        # Get the target labels by checking if source-destination pairs exist
        target = torch.isin(
            batch[task.dst_entity_table].batch
            + batch_size * batch[task.dst_entity_table].n_id,
            src_batch + batch_size * dst_index,
        ).float()  # Convert the result to float for loss computation

        # Optimization
        optimizer.zero_grad()  # Clear previous gradients
        loss = F.binary_cross_entropy_with_logits(out, target)  # Compute binary cross-entropy loss
        loss.backward()  # Backpropagation to compute gradients

        optimizer.step()  # Update model parameters

        # Accumulate the total loss and count for averaging later
        loss_accum += float(loss) * out.numel()
        count_accum += out.numel()

        steps += 1
        if steps >= total_steps:
            break  # Break the loop if max steps per epoch is reached

    # Handle the case where no data was sampled
    if count_accum == 0:
        warnings.warn(
            f"Did not sample a single '{task.dst_entity_table}' node in any mini-batch. "
            "Try increasing the number of layers/hops or reducing the batch size."
        )

    # Return average loss for the epoch
    return loss_accum / count_accum if count_accum > 0 else float("nan")


In [27]:
@torch.no_grad()  # No gradient computation for evaluation
def test(loader: NeighborLoader) -> np.ndarray:
    model.eval()  # Set model to evaluation mode

    pred_list: list[Tensor] = []  # Store predictions
    for batch in tqdm(loader):  # Iterate over batches in the test loader
        batch = batch.to(device)  # Move the batch data to the device (GPU or CPU)

        # Forward pass through the model for link prediction
        out = (
            model.forward_dst_readout(
                batch, task.src_entity_table, task.dst_entity_table
            )
            .detach()
            .flatten()  # Detach the output from the computational graph
        )

        batch_size = batch[task.src_entity_table].batch_size  # Get the batch size for source nodes

        # Prepare a tensor to hold the scores for the source-destination pairs
        scores = torch.zeros(batch_size, task.num_dst_nodes, device=out.device)

        # Fill the scores with sigmoid activations for the destination nodes in the current batch
        scores[
            batch[task.dst_entity_table].batch, batch[task.dst_entity_table].n_id
        ] = torch.sigmoid(out)  # Apply sigmoid activation to get probabilities

        # Use top-k (e.g., top recommended items) based on the scores
        _, pred_mini = torch.topk(scores, k=task.eval_k, dim=1)  # Get top-k predictions
        pred_list.append(pred_mini)  # Append predictions to the list

    # Concatenate all predictions and move to CPU for further processing
    pred = torch.cat(pred_list, dim=0).cpu().numpy()

    return pred  # Return the final predictions as a NumPy array


In [56]:
def test(loader: NeighborLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()  # Set the model to evaluation mode

    src_nodes_list: list[Tensor] = []  # Store source nodes (e.g., users)
    pred_articles_list: list[Tensor] = []  # Store predicted articles (edges)

    for batch in tqdm(loader):  # Loop through batches in the test loader
        batch = batch.to(device)  # Move batch to device (GPU/CPU)

        # Forward pass to get predictions from the model
        out = (
            model.forward_dst_readout(
                batch, task.src_entity_table, task.dst_entity_table
            )
            .detach()
            .flatten()  # Detach the output from the graph and flatten
        )

        batch_size = batch[task.src_entity_table].batch_size  # Get batch size of source nodes
        scores = torch.zeros(batch_size, task.num_dst_nodes, device=out.device)  # Initialize scores tensor

        # Populate scores for the destination nodes (articles)
        scores[
            batch[task.dst_entity_table].batch, batch[task.dst_entity_table].n_id
        ] = torch.sigmoid(out)  # Apply sigmoid to get probabilities

        # Get top-K predictions (e.g., top articles for each source node)
        _, top_articles = torch.topk(scores, k=task.eval_k, dim=1)  # Get top-K articles

        # Collect the source nodes (user) and their corresponding predicted articles (items)
        src_nodes_list.append(batch[task.src_entity_table].batch)  # Source node (users)
        pred_articles_list.append(top_articles)  # Predicted articles (items)

    # Concatenate all the source nodes and predictions
    src_nodes = torch.cat(src_nodes_list, dim=0).cpu().numpy()  # Convert to numpy
    pred_articles = torch.cat(pred_articles_list, dim=0).cpu().numpy()  # Convert to numpy

    return src_nodes, pred_articles  # Return source nodes and their predicted articles

In [43]:
# Assuming you have train_table, val_table, test_table already initialized with customer_id and article_id columns
customer_id_map = train_table.df[['customers_id']].reset_index(drop=True)
article_id_map = train_table.df[['articles_id']].reset_index(drop=True)


In [50]:
@torch.no_grad()
def test(loader: NeighborLoader) -> pd.DataFrame:
    model.eval()

    predictions = []  # List to store final predictions (customer_id, article_id pairs)

    for batch in tqdm(loader):
        batch = batch.to(device)

        # Forward pass to get model outputs
        out = model.forward_dst_readout(
            batch, task.src_entity_table, task.dst_entity_table
        ).detach().flatten()

        batch_size = batch[task.src_entity_table].batch_size
        scores = torch.zeros(batch_size, task.num_dst_nodes, device=out.device)

        # Apply sigmoid and fill the scores
        scores[batch[task.dst_entity_table].batch, batch[task.dst_entity_table].n_id] = torch.sigmoid(out)

        # Top-k prediction
        _, pred_mini = torch.topk(scores, k=task.eval_k, dim=1)

        # Get source (customers) and predicted destination nodes (articles)
        src_nodes = batch[task.src_entity_table].batch.cpu().numpy()  # Source nodes (e.g., customers)
        pred_articles = pred_mini.cpu().numpy()  # Predicted destination nodes (e.g., articles)

        # Map the source and destination nodes back to customer_id and article_id
        for src_node, articles in zip(src_nodes, pred_articles):
            customer_id = customer_id_map.iloc[src_node].values[0]  # Map src_node to customer_id
            for article_node in articles:
                article_id = article_id_map.iloc[article_node].values[0]  # Map article_node to article_id
                predictions.append({'customers_id': customer_id, 'articles_id': article_id})

    # Convert the predictions to a pandas DataFrame for further processing or saving as CSV
    return pd.DataFrame(predictions)


In [51]:
import copy

# Initialize variables for tracking the best model and best validation metrics
state_dict = None  # This will hold the best model state
best_val_metric = 0  # This will store the best validation metric
epochs = 10  # Set the number of epochs (you can adjust as needed)
eval_epochs_interval = 1  # Evaluate every 'n' epochs (change this based on your needs)
tune_metric = "link_prediction_map"  # Define the metric you are tuning

# Training and evaluation loop
for epoch in range(1, epochs + 1):
    # Run the training function
    train_loss = train()
    
    # Perform evaluation every 'eval_epochs_interval' epochs
    if epoch % eval_epochs_interval == 0:
        # Run the validation on the validation dataset
        val_pred = test(loader_dict["val"])  # Get the predictions from the model
        val_metrics = task.evaluate(val_pred, task.get_table("val"))  # Evaluate predictions
        
        # Print the training loss and validation metrics
        print(
            f"Epoch: {epoch:02d}, Train loss: {train_loss}, "
            f"Val metrics: {val_metrics}"
        )

        # Check if the current validation metric is the best
        if val_metrics[tune_metric] > best_val_metric:
            best_val_metric = val_metrics[tune_metric]  # Update best metric
            state_dict = copy.deepcopy(model.state_dict())  # Save the best model state

# After training, load the best model weights
model.load_state_dict(state_dict)

# Evaluate the model on the test set and get the predictions with customer_id and article_id
test_predictions = test(loader_dict["test"])

# Save the test predictions to a CSV file
test_predictions.to_csv('predicted_links.csv', index=False)

print(f"Test predictions saved to 'predicted_links.csv'")

100%|██████████| 30/30 [00:06<00:00,  4.31it/s]


ValueError: The shape of pred must be (15091, 10), but (150910, 2) given.

In [28]:
import copy

# Initialize variables for tracking the best model and best validation metrics
state_dict = None  # This will hold the best model state
best_val_metric = 0  # This will store the best validation metric
epochs = 10  # Set the number of epochs (you can adjust as needed)
eval_epochs_interval = 1  # Evaluate every 'n' epochs (change this based on your needs)
tune_metric = "link_prediction_map"  # Define the metric you are tuning

# Training and evaluation loop
for epoch in range(1, epochs + 1):
    # Run the training function
    train_loss = train()
    
    # Perform evaluation every 'eval_epochs_interval' epochs
    if epoch % eval_epochs_interval == 0:
        # Run the validation on the validation dataset
        val_pred = test(loader_dict["val"])  # Get the predictions from the model
        val_metrics = task.evaluate(val_pred, task.get_table("val"))  # Evaluate predictions
        
        # Print the training loss and validation metrics
        print(
            f"Epoch: {epoch:02d}, Train loss: {train_loss}, "
            f"Val metrics: {val_metrics}"
        )

        # Check if the current validation metric is the best
        if val_metrics[tune_metric] > best_val_metric:
            best_val_metric = val_metrics[tune_metric]  # Update best metric
            state_dict = copy.deepcopy(model.state_dict())  # Save the best model state

# After training, load the best model weights
model.load_state_dict(state_dict)

# Evaluate the model on the validation set with the best weights
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, task.get_table("val"))
print(f"Best Val metrics: {val_metrics}")

# Evaluate the model on the test set with the best weights
test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

100%|██████████| 146/146 [00:12<00:00, 11.35it/s]


Epoch: 01, Train loss: 0.04057414997108877, Val metrics: {'link_prediction_precision': np.float64(0.00795619622304168), 'link_prediction_recall': np.float64(0.04072652307962117), 'link_prediction_map': np.float64(0.025772221736840146)}


100%|██████████| 146/146 [00:12<00:00, 11.52it/s]


Epoch: 02, Train loss: 0.03814271093428482, Val metrics: {'link_prediction_precision': np.float64(0.00793049502737736), 'link_prediction_recall': np.float64(0.040631338093455927), 'link_prediction_map': np.float64(0.02580221720632704)}


100%|██████████| 146/146 [00:12<00:00, 11.57it/s]


...

Epoch: 10, Train loss: 0.03753233295115204, Val metrics: {'link_prediction_precision': np.float64(0.00802994748016538), 'link_prediction_recall': np.float64(0.04091593634757133), 'link_prediction_map': np.float64(0.0262047981863829)}


100%|██████████| 146/146 [00:12<00:00, 11.80it/s]


Best Val metrics: {'link_prediction_precision': np.float64(0.008044474242932169), 'link_prediction_recall': np.float64(0.040947062218158484), 'link_prediction_map': np.float64(0.02620739260808796)}


100%|██████████| 132/132 [00:11<00:00, 11.50it/s]


Best test metrics: {'link_prediction_precision': np.float64(0.0083986159100838), 'link_prediction_recall': np.float64(0.04430430252273792), 'link_prediction_map': np.float64(0.027922652595277084)}


In [23]:
import copy

# Initialize variables for tracking the best model and best validation metrics
state_dict = None  # This will hold the best model state
best_val_metric = 0  # This will store the best validation metric
epochs = 10  # Set the number of epochs (you can adjust as needed)
eval_epochs_interval = 1  # Evaluate every 'n' epochs (change this based on your needs)
tune_metric = "link_prediction_map"  # Define the metric you are tuning

# Training and evaluation loop
for epoch in range(1, epochs + 1):
    # Run the training function
    train_loss = train()
    
    # Perform evaluation every 'eval_epochs_interval' epochs
    if epoch % eval_epochs_interval == 0:
        # Run the validation on the validation dataset
        val_pred = test(loader_dict["val"])  # Get the predictions from the model
        val_metrics = task.evaluate(val_pred, task.get_table("val"))  # Evaluate predictions
        
        # Print the training loss and validation metrics
        print(
            f"Epoch: {epoch:02d}, Train loss: {train_loss}, "
            f"Val metrics: {val_metrics}"
        )

        # Check if the current validation metric is the best
        if val_metrics[tune_metric] > best_val_metric:
            best_val_metric = val_metrics[tune_metric]  # Update best metric
            state_dict = copy.deepcopy(model.state_dict())  # Save the best model state

# After training, load the best model weights
model.load_state_dict(state_dict)

# Evaluate the model on the validation set with the best weights
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, task.get_table("val"))
print(f"Best Val metrics: {val_metrics}")

# Evaluate the model on the test set with the best weights
test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

100%|██████████| 146/146 [00:13<00:00, 10.90it/s]


Epoch: 01, Train loss: 0.03945297533595412, Val metrics: {'link_prediction_precision': np.float64(0.007957313666331433), 'link_prediction_recall': np.float64(0.0407696419437334), 'link_prediction_map': np.float64(0.02562915838223258)}


100%|██████████| 146/146 [00:13<00:00, 11.07it/s]


Epoch: 02, Train loss: 0.03802653583146912, Val metrics: {'link_prediction_precision': np.float64(0.007907028718292547), 'link_prediction_recall': np.float64(0.040497773836788514), 'link_prediction_map': np.float64(0.02570407753243621)}


100%|██████████| 146/146 [00:13<00:00, 11.01it/s]


...

Epoch: 10, Train loss: 0.037548237429148534, Val metrics: {'link_prediction_precision': np.float64(0.008045591686221924), 'link_prediction_recall': np.float64(0.040971850365046164), 'link_prediction_map': np.float64(0.026211495537451206)}


100%|██████████| 146/146 [00:12<00:00, 11.84it/s]


Best Val metrics: {'link_prediction_precision': np.float64(0.008045591686221924), 'link_prediction_recall': np.float64(0.040971850365046164), 'link_prediction_map': np.float64(0.026211495537451206)}


100%|██████████| 132/132 [00:11<00:00, 11.34it/s]


Best test metrics: {'link_prediction_precision': np.float64(0.008377516978434408), 'link_prediction_recall': np.float64(0.04419156041473547), 'link_prediction_map': np.float64(0.02764693020184815)}


In [117]:
import copy

# Initialize variables for tracking the best model and best validation metrics
state_dict = None  # This will hold the best model state
best_val_metric = 0  # This will store the best validation metric
epochs = 10  # Set the number of epochs (you can adjust as needed)
eval_epochs_interval = 1  # Evaluate every 'n' epochs (change this based on your needs)
tune_metric = "link_prediction_map"  # Define the metric you are tuning

# Training and evaluation loop
for epoch in range(1, epochs + 1):
    # Run the training function
    train_loss = train()
    
    # Perform evaluation every 'eval_epochs_interval' epochs
    if epoch % eval_epochs_interval == 0:
        # Run the validation on the validation dataset
        val_pred = test(loader_dict["val"])  # Get the predictions from the model
        val_metrics = task.evaluate(val_pred, task.get_table("val"))  # Evaluate predictions
        
        # Print the training loss and validation metrics
        print(
            f"Epoch: {epoch:02d}, Train loss: {train_loss}, "
            f"Val metrics: {val_metrics}"
        )

        # Check if the current validation metric is the best
        if val_metrics[tune_metric] > best_val_metric:
            best_val_metric = val_metrics[tune_metric]  # Update best metric
            state_dict = copy.deepcopy(model.state_dict())  # Save the best model state

# After training, load the best model weights
model.load_state_dict(state_dict)

# Evaluate the model on the validation set with the best weights
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, task.get_table("val"))
print(f"Best Val metrics: {val_metrics}")

# Evaluate the model on the test set with the best weights
test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

100%|██████████| 30/30 [00:03<00:00,  8.52it/s]


Epoch: 01, Train loss: 0.17809573030820983, Val metrics: {'link_prediction_precision': np.float64(0.11182161553243655), 'link_prediction_recall': np.float64(0.08025433414899445), 'link_prediction_map': np.float64(0.10185752729735897), 'link_prediction_top': np.float64(0.3192631369690544)}


100%|██████████| 30/30 [00:03<00:00,  8.55it/s]


Epoch: 02, Train loss: 0.17688217352526944, Val metrics: {'link_prediction_precision': np.float64(0.11200384335034126), 'link_prediction_recall': np.float64(0.07955159331311874), 'link_prediction_map': np.float64(0.10129795904843947), 'link_prediction_top': np.float64(0.3206546948512358)}


100%|██████████| 30/30 [00:03<00:00,  8.48it/s]


...

Epoch: 10, Train loss: 0.17289290147091232, Val metrics: {'link_prediction_precision': np.float64(0.1162116493274137), 'link_prediction_recall': np.float64(0.08207204997860286), 'link_prediction_map': np.float64(0.10598986518822846), 'link_prediction_top': np.float64(0.3290703068053807)}


100%|██████████| 30/30 [00:03<00:00,  8.65it/s]


Best Val metrics: {'link_prediction_precision': np.float64(0.1162116493274137), 'link_prediction_recall': np.float64(0.08207204997860286), 'link_prediction_map': np.float64(0.10598986518822846), 'link_prediction_top': np.float64(0.3290703068053807)}


100%|██████████| 31/31 [00:03<00:00,  8.57it/s]


Best test metrics: {'link_prediction_precision': np.float64(0.11612083386951487), 'link_prediction_recall': np.float64(0.082524470940949), 'link_prediction_map': np.float64(0.10602989748209153), 'link_prediction_top': np.float64(0.32923690644704673)}


In [ ]:
import copy

# Initialize variables for tracking the best model and best validation metrics
state_dict = None  # This will hold the best model state
best_val_metric = 0  # This will store the best validation metric
epochs = 10  # Set the number of epochs (you can adjust as needed)
eval_epochs_interval = 1  # Evaluate every 'n' epochs (change this based on your needs)
tune_metric = "link_prediction_map"  # Define the metric you are tuning

# Training and evaluation loop
for epoch in range(1, epochs + 1):
    # Run the training function
    train_loss = train()
    
    # Perform evaluation every 'eval_epochs_interval' epochs
    if epoch % eval_epochs_interval == 0:
        # Run the validation on the validation dataset
        val_pred = test(loader_dict["val"])  # Get the predictions from the model
        val_metrics = task.evaluate(val_pred, task.get_table("val"))  # Evaluate predictions
        
        # Print the training loss and validation metrics
        print(
            f"Epoch: {epoch:02d}, Train loss: {train_loss}, "
            f"Val metrics: {val_metrics}"
        )

        # Check if the current validation metric is the best
        if val_metrics[tune_metric] > best_val_metric:
            best_val_metric = val_metrics[tune_metric]  # Update best metric
            state_dict = copy.deepcopy(model.state_dict())  # Save the best model state

# After training, load the best model weights
model.load_state_dict(state_dict)

# Evaluate the model on the validation set with the best weights
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, task.get_table("val"))
print(f"Best Val metrics: {val_metrics}")

# Evaluate the model on the test set with the best weights
test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

In [117]:
import copy

# Initialize variables for tracking the best model and best validation metrics
state_dict = None  # This will hold the best model state
best_val_metric = 0  # This will store the best validation metric
epochs = 10  # Set the number of epochs (you can adjust as needed)
eval_epochs_interval = 1  # Evaluate every 'n' epochs (change this based on your needs)
tune_metric = "link_prediction_map"  # Define the metric you are tuning

# Training and evaluation loop
for epoch in range(1, epochs + 1):
    # Run the training function
    train_loss = train()
    
    # Perform evaluation every 'eval_epochs_interval' epochs
    if epoch % eval_epochs_interval == 0:
        # Run the validation on the validation dataset
        val_pred = test(loader_dict["val"])  # Get the predictions from the model
        val_metrics = task.evaluate(val_pred, task.get_table("val"))  # Evaluate predictions
        
        # Print the training loss and validation metrics
        print(
            f"Epoch: {epoch:02d}, Train loss: {train_loss}, "
            f"Val metrics: {val_metrics}"
        )

        # Check if the current validation metric is the best
        if val_metrics[tune_metric] > best_val_metric:
            best_val_metric = val_metrics[tune_metric]  # Update best metric
            state_dict = copy.deepcopy(model.state_dict())  # Save the best model state

# After training, load the best model weights
model.load_state_dict(state_dict)

# Evaluate the model on the validation set with the best weights
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, task.get_table("val"))
print(f"Best Val metrics: {val_metrics}")

# Evaluate the model on the test set with the best weights
test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

100%|██████████| 30/30 [00:08<00:00,  3.47it/s]


Epoch: 01, Train loss: 0.18509238791182506, Val metrics: {'link_prediction_precision': np.float64(0.08001457822543237), 'link_prediction_recall': np.float64(0.06321641491657157), 'link_prediction_map': np.float64(0.0692774022780318)}


100%|██████████| 30/30 [00:08<00:00,  3.41it/s]


Epoch: 02, Train loss: 0.18468293568974875, Val metrics: {'link_prediction_precision': np.float64(0.07883838049168379), 'link_prediction_recall': np.float64(0.06231386771264629), 'link_prediction_map': np.float64(0.06567057996303904)}


100%|██████████| 30/30 [00:08<00:00,  3.52it/s]


...

Epoch: 10, Train loss: 0.18337202866440802, Val metrics: {'link_prediction_precision': np.float64(0.08115764362865284), 'link_prediction_recall': np.float64(0.06439167356594618), 'link_prediction_map': np.float64(0.06836626318850822)}


100%|██████████| 30/30 [00:08<00:00,  3.43it/s]


Best Val metrics: {'link_prediction_precision': np.float64(0.07996487972964018), 'link_prediction_recall': np.float64(0.0633939957823926), 'link_prediction_map': np.float64(0.06951899218813273)}


100%|██████████| 31/31 [00:09<00:00,  3.24it/s]


Best test metrics: {'link_prediction_precision': np.float64(0.08025028953802599), 'link_prediction_recall': np.float64(0.06395330736194382), 'link_prediction_map': np.float64(0.06759997998255624)}


In [133]:
import copy

# Initialize variables for tracking the best model and best validation metrics
state_dict = None  # This will hold the best model state
best_val_metric = 0  # This will store the best validation metric
epochs = 10  # Set the number of epochs (you can adjust as needed)
eval_epochs_interval = 1  # Evaluate every 'n' epochs (change this based on your needs)
tune_metric = "link_prediction_map"  # Define the metric you are tuning

# Training and evaluation loop
for epoch in range(1, epochs + 1):
    # Run the training function
    train_loss = train()
    
    # Perform evaluation every 'eval_epochs_interval' epochs
    if epoch % eval_epochs_interval == 0:
        # Run the validation on the validation dataset
        val_pred = test(loader_dict["val"])  # Get the predictions from the model
        val_metrics = task.evaluate(val_pred, task.get_table("val"))  # Evaluate predictions
        
        # Print the training loss and validation metrics
        print(
            f"Epoch: {epoch:02d}, Train loss: {train_loss}, "
            f"Val metrics: {val_metrics}"
        )

        # Check if the current validation metric is the best
        if val_metrics[tune_metric] > best_val_metric:
            best_val_metric = val_metrics[tune_metric]  # Update best metric
            state_dict = copy.deepcopy(model.state_dict())  # Save the best model state

# After training, load the best model weights
model.load_state_dict(state_dict)

# Evaluate the model on the validation set with the best weights
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, task.get_table("val"))
print(f"Best Val metrics: {val_metrics}")

# Evaluate the model on the test set with the best weights
test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

100%|██████████| 30/30 [00:08<00:00,  3.53it/s]


Epoch: 01, Train loss: 0.18943093903756822, Val metrics: {'link_prediction_precision': np.float64(0.07721489629580544), 'link_prediction_recall': np.float64(0.060872931022256035), 'link_prediction_map': np.float64(0.06538665429726327)}


100%|██████████| 30/30 [00:08<00:00,  3.53it/s]


Epoch: 02, Train loss: 0.18634521801331258, Val metrics: {'link_prediction_precision': np.float64(0.07913657146643695), 'link_prediction_recall': np.float64(0.06280025032600456), 'link_prediction_map': np.float64(0.06821440667358764)}


100%|██████████| 30/30 [00:08<00:00,  3.54it/s]


...

Epoch: 10, Train loss: 0.18363606004267352, Val metrics: {'link_prediction_precision': np.float64(0.08021337220860116), 'link_prediction_recall': np.float64(0.06337757732780143), 'link_prediction_map': np.float64(0.06640409294723124)}


100%|██████████| 30/30 [00:08<00:00,  3.58it/s]


Best Val metrics: {'link_prediction_precision': np.float64(0.08100854814127625), 'link_prediction_recall': np.float64(0.06426432493441359), 'link_prediction_map': np.float64(0.06929534895706785)}


100%|██████████| 31/31 [00:09<00:00,  3.14it/s]


Best test metrics: {'link_prediction_precision': np.float64(0.08195534680221336), 'link_prediction_recall': np.float64(0.06486584500684586), 'link_prediction_map': np.float64(0.06861559716324225)}


In [170]:
import os
import copy
import json
import warnings
import torch
import torch.nn.functional as F
# from model import Model
from torch import Tensor
from torch_geometric.loader import NeighborLoader
from tqdm import tqdm
from relbench.base import Dataset, RecommendationTask
from relbench.modeling.graph import get_link_train_table_input, make_pkey_fkey_graph
from relbench.modeling.loader import LinkNeighborLoader
from relbench.tasks import get_task
from pathlib import Path

# Define static configuration parameters
learning_rate = 0.001
epochs = 20
eval_epochs_interval = 1
batch_size = 512
channels = 128
aggregation_method = "sum"
num_layers = 2
num_neighbors = 128
temporal_strategy = "uniform"
share_same_time = True
max_steps_per_epoch = 2000
seed = 42



In [171]:
# Set up the number of neighbors for NeighborLoader
num_neighbors = [num_neighbors // 2**i for i in range(num_layers)]

# Prepare training data
train_table_input = get_link_train_table_input(task.get_table("train"), task)
train_loader = LinkNeighborLoader(
    data=data,
    num_neighbors=num_neighbors,
    time_attr="time",
    src_nodes=train_table_input.src_nodes,
    dst_nodes=train_table_input.dst_nodes,
    num_dst_nodes=train_table_input.num_dst_nodes,
    src_time=train_table_input.src_time,
    share_same_time=share_same_time,
    batch_size=batch_size,
    temporal_strategy=temporal_strategy,
    shuffle=not share_same_time,
    num_workers=0,
)

# Set up evaluation data loaders for validation and test splits
eval_loaders_dict: Dict[str, Tuple[NeighborLoader, NeighborLoader]] = {}
for split in ["val", "test"]:
    timestamp = dataset.val_timestamp if split == "val" else dataset.test_timestamp
    seed_time = int(timestamp.timestamp())
    target_table = task.get_table(split)
    
    src_node_indices = torch.from_numpy(target_table.df[task.src_entity_col].values)
    src_loader = NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        time_attr="time",
        input_nodes=(task.src_entity_table, src_node_indices),
        input_time=torch.full(size=(len(src_node_indices),), fill_value=seed_time, dtype=torch.long),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    dst_loader = NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        time_attr="time",
        input_nodes=task.dst_entity_table,
        input_time=torch.full(size=(task.num_dst_nodes,), fill_value=seed_time, dtype=torch.long),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    eval_loaders_dict[split] = (src_loader, dst_loader)


AttributeError: 'CustomerChurnTask' object has no attribute 'src_entity_col'

In [45]:
# Initialize the model
model = Model(
    data=data,
    col_stats_dict=col_stats_dict,
    num_layers=num_layers,
    channels=channels,
    out_channels=channels,
    aggr=aggregation_method,
    norm="layer_norm",
    shallow_list=[task.dst_entity_table] if share_same_time else [],
).to(device)

# Set up the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [47]:
def train() -> float:
    model.train()
    loss_accum = count_accum = 0
    steps = 0
    total_steps = min(len(train_loader), max_steps_per_epoch)
    
    for batch in tqdm(train_loader, total=total_steps):
        src_batch, batch_pos_dst, batch_neg_dst = batch
        src_batch, batch_pos_dst, batch_neg_dst = (
            src_batch.to(device),
            batch_pos_dst.to(device),
            batch_neg_dst.to(device),
        )
        
        # Forward pass
        x_src = model(src_batch, task.src_entity_table)
        x_pos_dst = model(batch_pos_dst, task.dst_entity_table)
        x_neg_dst = model(batch_neg_dst, task.dst_entity_table)
        
        pos_score = torch.sum(x_src * x_pos_dst, dim=1)
        if share_same_time:
            neg_score = x_src @ x_neg_dst.t()
            pos_score = pos_score.view(-1, 1)
        else:
            neg_score = torch.sum(x_src * x_neg_dst, dim=1)
        
        # Compute loss
        optimizer.zero_grad()
        diff_score = pos_score - neg_score
        loss = F.softplus(-diff_score).mean()
        loss.backward()
        optimizer.step()
        
        loss_accum += float(loss) * x_src.size(0)
        count_accum += x_src.size(0)
        steps += 1
        if steps >= max_steps_per_epoch:
            break
    
    if count_accum == 0:
        warnings.warn("No nodes sampled in the mini-batch.")
    
    return loss_accum / count_accum if count_accum > 0 else float("nan")


In [48]:
@torch.no_grad()
def test(src_loader: NeighborLoader, dst_loader: NeighborLoader) -> np.ndarray:
    model.eval()
    
    dst_embs: list[Tensor] = []
    for batch in tqdm(dst_loader):
        batch = batch.to(device)
        emb = model(batch, task.dst_entity_table).detach()
        dst_embs.append(emb)
    dst_emb = torch.cat(dst_embs, dim=0)
    del dst_embs
    
    pred_index_mat_list: list[Tensor] = []
    for batch in tqdm(src_loader):
        batch = batch.to(device)
        emb = model(batch, task.src_entity_table)
        _, pred_index_mat = torch.topk(emb @ dst_emb.t(), k=task.eval_k, dim=1)
        pred_index_mat_list.append(pred_index_mat.cpu())
    
    pred = torch.cat(pred_index_mat_list, dim=0).numpy()
    return pred


In [49]:
state_dict = None
best_val_metric = 0

for epoch in range(1, epochs + 1):
    # Train the model
    train_loss = train()
    
    # Perform evaluation every eval_epochs_interval
    if epoch % eval_epochs_interval == 0:
        # Run validation
        val_pred = test(*eval_loaders_dict["val"])
        val_metrics = task.evaluate(val_pred, task.get_table("val"))
        print(f"Epoch: {epoch:02d}, Train loss: {train_loss}, Val metrics: {val_metrics}")
        
        # Save the best model based on the validation metric
        if val_metrics[tune_metric] >= best_val_metric:
            best_val_metric = val_metrics[tune_metric]
            state_dict = copy.deepcopy(model.state_dict())

# Load the best model and evaluate on validation and test sets
model.load_state_dict(state_dict)
val_pred = test(*eval_loaders_dict["val"])
val_metrics = task.evaluate(val_pred, task.get_table("val"))
print(f"Best Val metrics: {val_metrics}")

test_pred = test(*eval_loaders_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")


100%|██████████| 30/30 [00:05<00:00,  5.74it/s]


Epoch: 01, Train loss: 0.1881787126983313, Val metrics: {'link_prediction_precision': np.float64(0.047826519117354714), 'link_prediction_recall': np.float64(0.05060422783530925), 'link_prediction_map': np.float64(0.044536754798665874), 'link_prediction_top': np.float64(0.1742097939169041)}


100%|██████████| 30/30 [00:05<00:00,  5.74it/s]


Epoch: 02, Train loss: 0.15225879719526564, Val metrics: {'link_prediction_precision': np.float64(0.04007355377377245), 'link_prediction_recall': np.float64(0.03767186381247256), 'link_prediction_map': np.float64(0.03528455149868575), 'link_prediction_top': np.float64(0.14869789941024453)}


100%|██████████| 30/30 [00:05<00:00,  5.85it/s]


...

Epoch: 20, Train loss: 0.10960065666451771, Val metrics: {'link_prediction_precision': np.float64(0.057583990457888805), 'link_prediction_recall': np.float64(0.05569334701643092), 'link_prediction_map': np.float64(0.04784906750896414), 'link_prediction_top': np.float64(0.205685507918627)}


100%|██████████| 30/30 [00:04<00:00,  6.20it/s]


Best Val metrics: {'link_prediction_precision': np.float64(0.06573454376780863), 'link_prediction_recall': np.float64(0.06186472077355334), 'link_prediction_map': np.float64(0.05724530441249016), 'link_prediction_top': np.float64(0.22609502352395466)}


100%|██████████| 31/31 [00:05<00:00,  6.01it/s]


Best test metrics: {'link_prediction_precision': np.float64(0.07010037318234462), 'link_prediction_recall': np.float64(0.06372047971152364), 'link_prediction_map': np.float64(0.06111575801770114), 'link_prediction_top': np.float64(0.24334062540213613)}


In [37]:
import copy

# Initialize variables for tracking the best model and best validation metrics
state_dict = None  # This will hold the best model state
best_val_metric = 0  # This will store the best validation metric
epochs = 10  # Set the number of epochs (you can adjust as needed)
eval_epochs_interval = 1  # Evaluate every 'n' epochs (change this based on your needs)
tune_metric = "link_prediction_map"  # Define the metric you are tuning

# Training and evaluation loop
for epoch in range(1, epochs + 1):
    # Run the training function
    train_loss = train()
    
    # Perform evaluation every 'eval_epochs_interval' epochs
    if epoch % eval_epochs_interval == 0:
        # Run the validation on the validation dataset
        val_pred = test(loader_dict["val"])  # Get the predictions from the model
        val_metrics = task.evaluate(val_pred, task.get_table("val"))  # Evaluate predictions
        
        # Print the training loss and validation metrics
        print(
            f"Epoch: {epoch:02d}, Train loss: {train_loss}, "
            f"Val metrics: {val_metrics}"
        )

        # Check if the current validation metric is the best
        if val_metrics[tune_metric] > best_val_metric:
            best_val_metric = val_metrics[tune_metric]  # Update best metric
            state_dict = copy.deepcopy(model.state_dict())  # Save the best model state

# After training, load the best model weights
model.load_state_dict(state_dict)

# Evaluate the model on the validation set with the best weights
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, task.get_table("val"))
print(f"Best Val metrics: {val_metrics}")

# Evaluate the model on the test set with the best weights
test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

100%|██████████| 30/30 [00:03<00:00,  9.33it/s]


Epoch: 01, Train loss: 0.18935488479761253, Val metrics: {'link_prediction_precision': np.float64(0.05603339738917237), 'link_prediction_recall': np.float64(0.09454541316894656), 'link_prediction_map': np.float64(0.05558086707208818)}


100%|██████████| 30/30 [00:03<00:00,  9.27it/s]


Epoch: 02, Train loss: 0.18626386191135222, Val metrics: {'link_prediction_precision': np.float64(0.05719302895765689), 'link_prediction_recall': np.float64(0.09634474311277003), 'link_prediction_map': np.float64(0.05772493096972012)}


100%|██████████| 30/30 [00:03<00:00,  9.28it/s]


...

Epoch: 10, Train loss: 0.18368114313055145, Val metrics: {'link_prediction_precision': np.float64(0.058399045788880784), 'link_prediction_recall': np.float64(0.09643348844236113), 'link_prediction_map': np.float64(0.057254656229823946)}


100%|██████████| 30/30 [00:03<00:00,  9.40it/s]


Best Val metrics: {'link_prediction_precision': np.float64(0.05841229872109204), 'link_prediction_recall': np.float64(0.09694216374882242), 'link_prediction_map': np.float64(0.058371931319274976)}


100%|██████████| 31/31 [00:03<00:00,  9.41it/s]


Best test metrics: {'link_prediction_precision': np.float64(0.058512417964225974), 'link_prediction_recall': np.float64(0.09737462951637656), 'link_prediction_map': np.float64(0.058545388812097)}


In [115]:
import copy

# Initialize variables for tracking the best model and best validation metrics
state_dict = None  # This will hold the best model state
best_val_metric = 0  # This will store the best validation metric
epochs = 10  # Set the number of epochs (you can adjust as needed)
eval_epochs_interval = 1  # Evaluate every 'n' epochs (change this based on your needs)
tune_metric = "link_prediction_map"  # Define the metric you are tuning

# Training and evaluation loop
for epoch in range(1, epochs + 1):
    # Run the training function
    train_loss = train()
    
    # Perform evaluation every 'eval_epochs_interval' epochs
    if epoch % eval_epochs_interval == 0:
        # Run the validation on the validation dataset
        val_pred = test(loader_dict["val"])  # Get the predictions from the model
        val_metrics = task.evaluate(val_pred, task.get_table("val"))  # Evaluate predictions
        
        # Print the training loss and validation metrics
        print(
            f"Epoch: {epoch:02d}, Train loss: {train_loss}, "
            f"Val metrics: {val_metrics}"
        )

        # Check if the current validation metric is the best
        if val_metrics[tune_metric] > best_val_metric:
            best_val_metric = val_metrics[tune_metric]  # Update best metric
            state_dict = copy.deepcopy(model.state_dict())  # Save the best model state

# After training, load the best model weights
model.load_state_dict(state_dict)

# Evaluate the model on the validation set with the best weights
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, task.get_table("val"))
print(f"Best Val metrics: {val_metrics}")

# Evaluate the model on the test set with the best weights
test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")


100%|██████████| 30/30 [00:08<00:00,  3.41it/s]


AttributeError: 'tuple' object has no attribute 'shape'

In [83]:
src_nodes, pred_articles = test(loader_dict["test"])

# Print source nodes (users) and their corresponding predicted articles
for src_node, articles in zip(src_nodes, pred_articles):
    print(f"Source Node (User): {src_node}, Predicted Articles: {articles}")

100%|██████████| 31/31 [00:20<00:00,  1.50it/s]


Source Node (User): 0, Predicted Articles: [  869 15992   643  1500  5228    25   177  2076  8209   850]
Source Node (User): 1, Predicted Articles: [1653 1494 1649 4220 5822 4405    2    3    1    0]
Source Node (User): 2, Predicted Articles: [ 1500  4000  3059   380   505 17067  1995  1624   426  8163]
Source Node (User): 3, Predicted Articles: [7 6 4 5 1 0 2 3 8 9]
Source Node (User): 4, Predicted Articles: [15992   126   643    25   110   547 15736   934    16     7]
Source Node (User): 5, Predicted Articles: [ 601 2329  658  331  731  807 5426  504 3778  924]
Source Node (User): 6, Predicted Articles: [ 126 5417 1936 2645  443  934    7  911   45  426]
Source Node (User): 7, Predicted Articles: [  126   643    25   505   314  4759   536 13809 17277   255]
Source Node (User): 8, Predicted Articles: [  310   643  4000   192  2645 16263  8771  8209 16220    90]
Source Node (User): 9, Predicted Articles: [7 6 4 5 1 0 2 3 8 9]
Source Node (User): 10, Predicted Articles: [ 4000   523  20

In [84]:
data = {'Source Node (User)': src_nodes}

# Assuming top-K predictions, add predicted articles as separate columns in the DataFrame
for i in range(pred_articles.shape[1]):  # For each top-K article
    data[f'Predicted Article {i+1}'] = pred_articles[:, i]

# Convert the dictionary to a DataFrame
df = pd.DataFrame(data)

# Save the DataFrame to a CSV file
output_csv_file = 'predicted_edges.csv'  # You can modify the file name if needed
df.to_csv(output_csv_file, index=False)

# Print confirmation
print(f"CSV file with source nodes and predicted articles saved as {output_csv_file}")

CSV file with source nodes and predicted articles saved as predicted_edges.csv


In [85]:
test_table

Table(df=
       timestamp  customers_id  \
0     2022-02-22           810   
1     2022-02-22         55934   
2     2022-02-22         31213   
3     2022-02-22        288754   
4     2022-02-22         96646   
...          ...           ...   
15537 2022-02-22        292185   
15538 2022-02-22          6240   
15539 2022-02-22        120378   
15540 2022-02-22          3242   
15541 2022-02-22         27603   

                                             articles_id  
0      [2581, 84, 16572, 3597, 430, 4862, 18183, 2335...  
1                                     [526, 1661, 19210]  
2      [4812, 7138, 4835, 1272, 3047, 507, 4715, 2833...  
3      [934, 204, 5318, 238, 13195, 19374, 6629, 2267...  
4      [2239, 11, 1033, 1039, 936, 3268, 16, 3854, 37...  
...                                                  ...  
15537                                             [3059]  
15538                                            [19616]  
15539                                             

In [92]:
@torch.no_grad()
def test(loader: NeighborLoader, test_table_df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()

    src_nodes_list: list[Tensor] = []  # Store source nodes (users)
    pred_list: list[Tensor] = []  # Store predicted articles (items)
    
    for batch in tqdm(loader):
        batch = batch.to(device)
        
        # Forward pass to get predictions
        out = (
            model.forward_dst_readout(
                batch, task.src_entity_table, task.dst_entity_table
            )
            .detach()
            .flatten()
        )
        
        batch_size = batch[task.src_entity_table].batch_size
        scores = torch.zeros(batch_size, task.num_dst_nodes, device=out.device)

        # Populate scores for the destination nodes
        scores[
            batch[task.dst_entity_table].batch, batch[task.dst_entity_table].n_id
        ] = torch.sigmoid(out)

        # Get top-K predicted articles for each source node
        _, pred_mini = torch.topk(scores, k=task.eval_k, dim=1)

        # Store the source nodes and predicted articles
        src_nodes_list.append(batch[task.src_entity_table].batch)
        pred_list.append(pred_mini)

    # Concatenate source nodes and predicted articles
    src_nodes = torch.cat(src_nodes_list, dim=0).cpu().numpy()  # Source nodes
    pred_articles = torch.cat(pred_list, dim=0).cpu().numpy()  # Predicted articles

    # Map source nodes (indices) to actual customers_id from the test table DataFrame
    customers_id = test_table_df.loc[src_nodes, 'customers_id'].values

    return customers_id, pred_articles


In [93]:
# Assuming test_table_df is your test table DataFrame containing 'customers_id'
customers_id, pred_articles = test(loader_dict["test"], test_table_df)

# Print source nodes (customers_id) and their corresponding predicted articles
for customer, articles in zip(customers_id, pred_articles):
    print(f"Customer ID: {customer}, Predicted Articles: {articles}")


NameError: name 'test_table_df' is not defined

In [172]:
from relbench.modeling.graph import get_node_train_table_input, make_pkey_fkey_graph
from torch_geometric.loader import NeighborLoader

loader_dict = {}


for split, table in [
    ("train", train_table),
    ("val", val_table),
    ("test", test_table),
]:
    table_input = get_node_train_table_input(
        table=table,
        task=task,
    )
    entity_table = table_input.nodes[0]
    loader_dict[split] = NeighborLoader(
        data,
        num_neighbors=[
            128 for i in range(2)
        ],  # we sample subgraphs of depth 2, 128 neighbors per node.
        time_attr="time",
        input_nodes=table_input.nodes,
        input_time=table_input.time,
        transform=table_input.transform,
        batch_size=512,
        temporal_strategy="uniform",
        shuffle=split == "train",
        num_workers=0,
        persistent_workers=False,
    )

In [173]:
from torch.nn import BCEWithLogitsLoss
import copy
from typing import Any, Dict, List

import torch
from torch import Tensor
from torch.nn import Embedding, ModuleDict
from torch_frame.data.stats import StatType
from torch_geometric.data import HeteroData
from torch_geometric.nn import MLP
from torch_geometric.typing import NodeType

from relbench.modeling.nn import HeteroEncoder, HeteroGraphSAGE, HeteroTemporalEncoder


class Model(torch.nn.Module):

    def __init__(
        self,
        data: HeteroData,
        col_stats_dict: Dict[str, Dict[str, Dict[StatType, Any]]],
        num_layers: int,
        channels: int,
        out_channels: int,
        aggr: str,
        norm: str,
        # List of node types to add shallow embeddings to input
        shallow_list: List[NodeType] = [],
        # ID awareness
        id_awareness: bool = False,
    ):
        super().__init__()

        self.encoder = HeteroEncoder(
            channels=channels,
            node_to_col_names_dict={
                node_type: data[node_type].tf.col_names_dict
                for node_type in data.node_types
            },
            node_to_col_stats=col_stats_dict,
        )
        self.temporal_encoder = HeteroTemporalEncoder(
            node_types=[
                node_type for node_type in data.node_types if "time" in data[node_type]
            ],
            channels=channels,
        )
        self.gnn = HeteroGraphSAGE(
            node_types=data.node_types,
            edge_types=data.edge_types,
            channels=channels,
            aggr=aggr,
            num_layers=num_layers,
        )
        self.head = MLP(
            channels,
            out_channels=out_channels,
            norm=norm,
            num_layers=1,
        )
        self.embedding_dict = ModuleDict(
            {
                node: Embedding(data.num_nodes_dict[node], channels)
                for node in shallow_list
            }
        )

        self.id_awareness_emb = None
        if id_awareness:
            self.id_awareness_emb = torch.nn.Embedding(1, channels)
        self.reset_parameters()

    def reset_parameters(self):
        self.encoder.reset_parameters()
        self.temporal_encoder.reset_parameters()
        self.gnn.reset_parameters()
        self.head.reset_parameters()
        for embedding in self.embedding_dict.values():
            torch.nn.init.normal_(embedding.weight, std=0.1)
        if self.id_awareness_emb is not None:
            self.id_awareness_emb.reset_parameters()

    def forward(
        self,
        batch: HeteroData,
        entity_table: NodeType,
    ) -> Tensor:
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
            batch.num_sampled_nodes_dict,
            batch.num_sampled_edges_dict,
        )

        return self.head(x_dict[entity_table][: seed_time.size(0)])

    def forward_dst_readout(
        self,
        batch: HeteroData,
        entity_table: NodeType,
        dst_table: NodeType,
    ) -> Tensor:
        if self.id_awareness_emb is None:
            raise RuntimeError(
                "id_awareness must be set True to use forward_dst_readout"
            )
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)
        # Add ID-awareness to the root node
        x_dict[entity_table][: seed_time.size(0)] += self.id_awareness_emb.weight

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
        )

        return self.head(x_dict[dst_table])


model = Model(
    data=data,
    col_stats_dict=col_stats_dict,
    num_layers=2,
    channels=128,
    out_channels=1,
    aggr="sum",
    norm="batch_norm",
).to(device)


# if you try out different RelBench tasks you will need to change these
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
epochs = 10

In [174]:
def train() -> float:
    model.train()

    loss_accum = count_accum = 0
    for batch in tqdm(loader_dict["train"]):
        batch = batch.to(device)

        optimizer.zero_grad()
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred

        loss = loss_fn(pred.float(), batch[entity_table].y.float())
        loss.backward()
        optimizer.step()

        loss_accum += loss.detach().item() * pred.size(0)
        count_accum += pred.size(0)

    return loss_accum / count_accum


@torch.no_grad()
def test(loader: NeighborLoader) -> np.ndarray:
    model.eval()

    pred_list = []
    for batch in loader:
        batch = batch.to(device)
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred
        pred_list.append(pred.detach().cpu())
    return torch.cat(pred_list, dim=0).numpy()

In [175]:
state_dict = None
best_val_metric = -math.inf if higher_is_better else math.inf
for epoch in range(1, epochs + 1):
    train_loss = train()
    val_pred = test(loader_dict["val"])
    val_metrics = task.evaluate(val_pred, val_table)
    print(f"Epoch: {epoch:02d}, Train loss: {train_loss}, Val metrics: {val_metrics}")

    if (higher_is_better and val_metrics[tune_metric] > best_val_metric) or (
        not higher_is_better and val_metrics[tune_metric] < best_val_metric
    ):
        best_val_metric = val_metrics[tune_metric]
        state_dict = copy.deepcopy(model.state_dict())


model.load_state_dict(state_dict)
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, val_table)
print(f"Best Val metrics: {val_metrics}")

test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

100%|██████████| 2939/2939 [02:25<00:00, 20.24it/s]


Epoch: 01, Train loss: 0.2794271792647381, Val metrics: {'average_precision': np.float64(0.9785210703759605), 'accuracy': 0.8884318766066838, 'f1': np.float64(0.9391645640594337), 'roc_auc': np.float64(0.866161584231297)}


100%|██████████| 2939/2939 [02:27<00:00, 19.91it/s]


Epoch: 02, Train loss: 0.27304596723775965, Val metrics: {'average_precision': np.float64(0.9805036774219051), 'accuracy': 0.8865809768637533, 'f1': np.float64(0.9372332555625107), 'roc_auc': np.float64(0.8760502881660223)}


100%|██████████| 2939/2939 [02:28<00:00, 19.81it/s]


...

Epoch: 08, Train loss: 0.2678582989362305, Val metrics: {'average_precision': np.float64(0.9827776305787627), 'accuracy': 0.8880205655526993, 'f1': np.float64(0.9375394321766561), 'roc_auc': np.float64(0.8893546353324011)}


100%|██████████| 2939/2939 [02:28<00:00, 19.77it/s]


Epoch: 09, Train loss: 0.26769193776739886, Val metrics: {'average_precision': np.float64(0.9820967426074018), 'accuracy': 0.8798971722365039, 'f1': np.float64(0.9320297951582868), 'roc_auc': np.float64(0.8852918902862363)}


100%|██████████| 2939/2939 [02:28<00:00, 19.76it/s]


Epoch: 10, Train loss: 0.26728961117589173, Val metrics: {'average_precision': np.float64(0.981642206251161), 'accuracy': 0.868586118251928, 'f1': np.float64(0.9241813004271476), 'roc_auc': np.float64(0.8819909220943105)}
Best Val metrics: {'average_precision': np.float64(0.983456387961197), 'accuracy': 0.8885347043701799, 'f1': np.float64(0.9379862700228833), 'roc_auc': np.float64(0.8915154671242516)}
Best test metrics: {'average_precision': np.float64(0.9813158527382689), 'accuracy': 0.8806621793705658, 'f1': np.float64(0.9336368234699038), 'roc_auc': np.float64(0.8867273449655223)}


In [177]:
mapped_predictions = task.map_predictions_to_original_ids(test_pred, split="test")

# Now you can evaluate with the original IDs
task.evaluate(mapped_predictions, target_table=target_table)

AttributeError: 'CustomerChurnTask' object has no attribute 'map_predictions_to_original_ids'

In [125]:
model.load_state_dict(state_dict)
test_pred = test(loader_dict["test"])

In [176]:
print(test_pred)

[10.1081295  3.5734766  3.5172772 ...  6.2169504  1.7859852  7.3400965]


In [179]:
customer_ids = test_table.df["customers_id"].values  # Extract the customer_id column
predicted_labels = test_pred  # This contains the model's predicted labels

# Now zip customer_ids with the predictions
results = list(zip(customer_ids, predicted_labels))

# If you'd like to show or process the results, you can format them as a dataframe
results_df = pd.DataFrame(results, columns=["customers_id", "predicted_churn"])

# You can now inspect the predictions
print(results_df)

       customers_id  predicted_churn
0            820105        10.108130
1            820432         3.573477
2            820663         3.517277
3            822402         1.025771
4            824978         7.288265
...             ...              ...
10989        600080         2.724762
10990        603426         1.736018
10991        607081         6.216950
10992        608889         1.785985
10993        609116         7.340096

[10994 rows x 2 columns]


In [182]:
results_df.to_csv('C:\\Users\\KN2C\Desktop\\Dani\\relbench\\result.csv', index=False)

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\KN2C\AppData\Local\Temp\ipykernel_19272\3727152186.py:1: SyntaxWarning: invalid escape sequence '\D'
  results_df.to_csv('C:\\Users\\KN2C\Desktop\\Dani\\relbench\\result.csv', index=False)


In [115]:
def forward(
    self,
    batch: HeteroData,
    entity_table: NodeType,
) -> Tensor:
    seed_time = batch[entity_table].seed_time
    x_dict = self.encoder(batch.tf_dict)

    # Temporal encoding
    rel_time_dict = self.temporal_encoder(
        seed_time, batch.time_dict, batch.batch_dict
    )

    # Add temporal features to node representations
    for node_type, rel_time in rel_time_dict.items():
        x_dict[node_type] = x_dict[node_type] + rel_time

    # Add shallow embeddings
    for node_type, embedding in self.embedding_dict.items():
        x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

    # Apply GNN layers
    x_dict = self.gnn(
        x_dict,
        batch.edge_index_dict,
        batch.num_sampled_nodes_dict,
        batch.num_sampled_edges_dict,
    )

    # Get predictions for the entity_table (customer nodes)
    predictions = self.head(x_dict[entity_table][: seed_time.size(0)])

    # Ensure that the model only returns the prediction and customer IDs
    return predictions, batch[entity_table].n_id  # Return only predictions and customer IDs


In [116]:
@torch.no_grad()
def test(loader: NeighborLoader, entity_table: NodeType) -> List[Dict]:
    model.eval()

    predictions = []
    customer_ids = []
    
    for batch in loader:
        batch = batch.to(device)
        pred, customer_id = model(batch, entity_table)  # Get predictions and customer IDs
        
        # Flatten the predictions if needed
        pred = pred.view(-1) if pred.size(1) == 1 else pred
        predictions.append(pred.detach().cpu())
        customer_ids.append(customer_id.detach().cpu())

    # Concatenate predictions and customer_ids from all batches
    predictions = torch.cat(predictions, dim=0).numpy()
    customer_ids = torch.cat(customer_ids, dim=0).numpy()

    # Combine predictions and customer_ids into a list of dictionaries
    result = [{"customer_id": cid, "prediction": pred} for cid, pred in zip(customer_ids, predictions)]
    
    return result


In [117]:
# Example usage during testing:
test_results = test(loader_dict["test"], entity_table=task.entity_table)

# Print out the first few results
for result in test_results[:5]:  # Print first 5 customer predictions
    print(f"Customer ID: {result['customer_id']}, Churn Prediction: {result['prediction']}")


ValueError: too many values to unpack (expected 2)

In [74]:
# Imports
import os
import math
import numpy as np
import torch
from torch.nn import BCEWithLogitsLoss, L1Loss
from torch_geometric.loader import NeighborLoader
from torch_geometric.seed import seed_everything
from torch_geometric.data import HeteroData
from torch_geometric.nn import MLP
from tqdm import tqdm
import copy

from relbench.datasets import get_dataset
from relbench.tasks import get_task
from relbench.modeling.utils import get_stype_proposal
from relbench.modeling.graph import make_pkey_fkey_graph, get_node_train_table_input
from torch_frame.config.text_embedder import TextEmbedderConfig
from relbench.modeling.graph import get_link_train_table_input
# Set random seed
seed_everything(42)

# Setting up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Get dataset and task
dataset = get_dataset("hyper-aras")
task = get_task("hyper-aras", "aras_recom_task")

# Get train/val/test tables
train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test")

# Set up parameters
epochs = 10
out_channels = 1
loss_fn = L1Loss()  # Loss function
tune_metric = "link_prediction_map"  # Metric for evaluation
higher_is_better = True

# Load and prepare data
db = dataset.get_db()
col_to_stype_dict = get_stype_proposal(db)

# Text embedding configuration (Replace with Persian BERT for example)
text_embedder_cfg = TextEmbedderConfig(
    text_embedder=BertPersianTextEmbedding(device=device), batch_size=256
)

# Make graph data
data, col_stats_dict = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=col_to_stype_dict,
    text_embedder_cfg=text_embedder_cfg,
    cache_dir=os.path.join("data_ARAS", f"rel-aras_recom_materialized2_cache"),
)

loader_dict = {}

for split, table in [("train", train_table), ("val", val_table), ("test", test_table)]:
    table_input = get_link_train_table_input(
        table=table,
        task=task,
    )
    src_entity_table = table_input.src_nodes[0]  # Source entity (e.g., users)
    dst_entity_table = table_input.dst_nodes[0]  # Destination entity (e.g., items)
    print("src_nodes:", table_input.src_nodes)
    print("dst_nodes (sparse):", table_input.dst_nodes)
    print("num_dst_nodes:", table_input.num_dst_nodes)
    print("src_time:", table_input.src_time)

    # Ensure NeighborLoader is correctly initialized
    loader_dict[split] = NeighborLoader(
        data,
        num_neighbors=[128 for _ in range(2)],
        input_nodes=(table_input.src_nodes, table_input.dst_nodes),
        input_time=table_input.src_time if table_input.src_time is not None else None,  # Check for None
        batch_size=512,
        shuffle=split == "train",
        num_workers=0,
        persistent_workers=False,
    )

# Define model architecture
class Model(torch.nn.Module):
    def __init__(self, data, col_stats_dict, num_layers, channels, out_channels, aggr, norm):
        super().__init__()
        self.encoder = HeteroEncoder(
            channels=channels,
            node_to_col_names_dict={node_type: data[node_type].tf.col_names_dict for node_type in data.node_types},
            node_to_col_stats=col_stats_dict,
        )
        self.gnn = HeteroGraphSAGE(
            node_types=data.node_types,
            edge_types=data.edge_types,
            channels=channels,
            aggr=aggr,
            num_layers=num_layers,
        )
        self.head = MLP(
            channels,
            out_channels=out_channels,
            norm=norm,
            num_layers=1,
        )

    def forward(self, batch: HeteroData, entity_table: str) -> torch.Tensor:
        x_dict = self.encoder(batch.tf_dict)
        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
            batch.num_sampled_nodes_dict,
            batch.num_sampled_edges_dict,
        )
        return self.head(x_dict[entity_table][: batch[entity_table].seed_time.size(0)])

# Initialize the model
model = Model(
    data=data,
    col_stats_dict=col_stats_dict,
    num_layers=2,
    channels=128,
    out_channels=out_channels,
    aggr="sum",
    norm="batch_norm",
).to(device)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# Training function
def train() -> float:
    model.train()
    loss_accum = count_accum = 0
    for batch in tqdm(loader_dict["train"]):
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch, task.entity_table)
        loss = loss_fn(pred.float(), batch[task.entity_table].y.float())
        loss.backward()
        optimizer.step()
        loss_accum += loss.detach().item() * pred.size(0)
        count_accum += pred.size(0)
    return loss_accum / count_accum

# Test function
@torch.no_grad()
def test(loader: NeighborLoader) -> np.ndarray:
    model.eval()
    pred_list = []
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch, task.entity_table)
        pred_list.append(pred.detach().cpu())
    return torch.cat(pred_list, dim=0).numpy()

# Training loop
best_val_metric = -math.inf if higher_is_better else math.inf
for epoch in range(1, epochs + 1):
    train_loss = train()
    val_pred = test(loader_dict["val"])
    val_metrics = task.evaluate(val_pred, val_table)
    print(f"Epoch: {epoch:02d}, Train loss: {train_loss}, Val metrics: {val_metrics}")
    
    # Save the best model
    if (higher_is_better and val_metrics[tune_metric] > best_val_metric) or (
        not higher_is_better and val_metrics[tune_metric] < best_val_metric
    ):
        best_val_metric = val_metrics[tune_metric]
        best_model = copy.deepcopy(model.state_dict())

# Load the best model and test
model.load_state_dict(best_model)
test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")


Using device: cuda


Some weights of BertModel were not initialized from the model checkpoint at HooshvareLab/bert-fa-zwnj-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


src_nodes: ('customer', tensor([ 13371,  13386,  13425,  ...,  48171,  64910, 284769]))
dst_nodes (sparse): ('article', tensor(crow_indices=tensor([      0,      15,      29,  ..., 6318711,
                            6318712, 6318713]),
       col_indices=tensor([   14,   126,   206,  ...,  5235,   869, 10591]),
       values=tensor([True, True, True,  ..., True, True, True]),
       size=(720868, 20298), nnz=6318713, layout=torch.sparse_csr))
num_dst_nodes: 20298
src_time: tensor([1616457600, 1616457600, 1616457600,  ..., 1644278400, 1644278400,
        1644278400])


ValueError: Received conflicting 'input_time' and 'time_attr' arguments: 'input_time' is set while 'time_attr' is not set.